# Task 2a: Text summarization with small files with Titan Text Premier


In this notebook, you ingest a small string of text directly into the Amazon Bedrock API (using the Titan Text model) and instruct it to summarize the input text. You can apply this approach to summarize call transcripts, meeting transcripts, books, articles, blog posts, and other relevant content when the input text length is within the context size limits of the model.

## Task 2a.1: Environment setup

In this task, you set up your environment.

In [1]:
#Create a service client by name using the default session.
import json
import os
import sys
import warnings

import boto3
import botocore

warnings.filterwarnings('ignore')
module_path = ".."
sys.path.append(os.path.abspath(module_path))

bedrock_client = boto3.client('bedrock-runtime',region_name=os.environ.get("AWS_DEFAULT_REGION", None))



## Task 2a.2: Writing prompt with text to be summarized

In this task, you use a short passage of text with fewer tokens than the maximum length supported by the foundation model. As a sample input text for this lab, you use a paragraph from an [AWS blog post](https://aws.amazon.com/jp/blogs/machine-learning/announcing-new-tools-for-building-with-generative-ai-on-aws/) announcing Amazon Bedrock.

The prompt starts with an instruction `Please provide a summary of the following text.`. 

In [2]:
prompt_data = """

Please provide a summary of the following text:

AWS took all of that feedback from customers, and today we are excited to announce Amazon Bedrock, \
a new service that makes FMs from AI21 Labs, Anthropic, Stability AI, and Amazon accessible via an API. \
Bedrock is the easiest way for customers to build and scale generative AI-based applications using FMs, \
democratizing access for all builders. Bedrock will offer the ability to access a range of powerful FMs \
for text and images—including Amazons Titan FMs, which consist of two new LLMs we’re also announcing \
today—through a scalable, reliable, and secure AWS managed service. With Bedrock’s serverless experience, \
customers can easily find the right model for what they’re trying to get done, get started quickly, privately \
customize FMs with their own data, and easily integrate and deploy them into their applications using the AWS \
tools and capabilities they are familiar with, without having to manage any infrastructure (including integrations \
with Amazon SageMaker ML features like Experiments to test different models and Pipelines to manage their FMs at scale).
"""

## Task 2a.3: Creating request body with prompt and inference parameters 

In this task, you create the request body with the above prompt and inference parameters.

In [3]:
# request body
body = json.dumps({
    "inputText": prompt_data, 
    "textGenerationConfig":{
        "maxTokenCount":2048,
        "stopSequences":[],
        "temperature":0,
        "topP":0.9
        }
    }) 

## Task 2a.4: Invoke foundation model via Boto3

In this task, you send an API request to Amazon Bedrock specifying the request parameters: `modelId`, `accept`, and `contentType`. Following the provided prompt, the foundation model in Amazon Bedrock then summarizes the input text.

### Complete Output Generation

By default, the Amazon Bedrock service generates the entire summary for a given prompt in a single output. This can be slow if the model output contains many tokens. 

In [6]:
import boto3

bedrock = boto3.client(
    "bedrock",
    region_name=bedrock_client.meta.region_name
)

models = bedrock.list_foundation_models(
    byProvider="Amazon"
)

for model in models["modelSummaries"]:
    if "Titan Text" in model["modelName"]:
        print("Model Name:", model["modelName"])
        print("Model ID:", model["modelId"])
        print("----------------------")

Model Name: Titan Text Embeddings V2
Model ID: amazon.titan-embed-text-v2:0
----------------------


In [7]:
bedrock_client = boto3.client(
    'bedrock-runtime',
    region_name='us-east-1'
)

In [12]:
modelId = 'amazon.titan-text-premier-v1:0'

In [16]:
modelId = "us.amazon.nova-2-lite-v1:0"

response = bedrock_client.invoke_model(
    body=body,
    modelId=modelId,
    accept=accept,
    contentType=contentType
)

response_body = json.loads(response["body"].read())

outputText = response_body["output"]["message"]["content"][0]["text"]

print(outputText)

### Summary:

AWS has launched **Amazon Bedrock**, a new service designed to make **foundation models (FMs)** from leading AI companies — including AI21 Labs, Anthropic, Stability AI, and Amazon — easily accessible through a simple API. Bedrock aims to **democratize access** to powerful generative AI models for all developers, enabling them to **build and scale AI-driven applications** efficiently.

Key features of Amazon Bedrock include:

- **Access to a variety of FMs** for both **text and image generation**, including **Amazon’s new Titan FMs** — two newly announced large language models (LLMs).
- **Serverless experience**: Users can **quickly select and deploy models** without managing infrastructure.
- **Private customization**: Developers can **tailor models using their own data** while maintaining privacy.
- **Seamless integration** with familiar **AWS tools** such as **Amazon SageMaker**, including features like **Experiments** for model testing and **Pipelines** for managing m

In [17]:
modelId = "us.amazon.nova-2-lite-v1:0"

accept = "application/json"
contentType = "application/json"

response = bedrock_client.invoke_model(
    body=body,
    modelId=modelId,
    accept=accept,
    contentType=contentType
)

In [18]:
response_body = json.loads(response["body"].read())

outputText = response_body["output"]["message"]["content"][0]["text"]

print(outputText)

### Summary

Amazon Web Services (AWS) has launched **Amazon Bedrock**, a new service designed to make **foundation models (FMs)** from leading AI companies — including AI21 Labs, Anthropic, Stability AI, and Amazon — easily accessible through a simple API. 

**Key Highlights of Amazon Bedrock:**

1. **Democratizing Access:**  
   Bedrock enables developers of all skill levels to build and scale **generative AI applications** using powerful FMs, removing the barriers to entry for AI-driven development.

2. **Model Variety:**  
   The service provides access to a range of advanced models for **text and image generation**, including **Amazon’s new Titan FMs** — two newly announced large language models (LLMs).

3. **Fully Managed Service:**  
   Bedrock is a **scalable, reliable, and secure** AWS-managed service, eliminating the need for customers to handle infrastructure.

4. **Serverless Experience:**  
   Users can:
   - Quickly find and select the right model for their use case.
   -

In [20]:
# Model configuration and invoke the model

modelId = "us.amazon.nova-2-lite-v1:0"
accept = "application/json"
contentType = "application/json"

outputText = "\n"

try:

    response = bedrock_client.invoke_model(
        body=body,
        modelId=modelId,
        accept=accept,
        contentType=contentType
    )

    response_body = json.loads(response.get('body').read())

    outputText = response_body.get('output', {}) \
                              .get('message', {}) \
                              .get('content', [{}])[0] \
                              .get('text', '')

except botocore.exceptions.ClientError as error:

    if error.response['Error']['Code'] == 'AccessDeniedException':

        print(f"\x1b[41m{error.response['Error']['Message']}\
        \nPlease check your AWS IAM permissions.\x1b[0m\n")

    else:
        raise error

print(outputText)

### Summary

Amazon Web Services (AWS) has launched **Amazon Bedrock**, a new service designed to make **foundation models (FMs)** from leading AI companies — including AI21 Labs, Anthropic, Stability AI, and Amazon — easily accessible via an API. Bedrock aims to **democratize access** to powerful generative AI models for all developers, enabling them to **build and scale AI-driven applications** efficiently.

Key features of Amazon Bedrock include:

- **Access to a variety of FMs**: Users can leverage models for both **text and image generation**, including **Amazon’s new Titan FMs** — two newly announced large language models (LLMs).
- **Fully managed service**: Bedrock is a **scalable, reliable, and secure** AWS service that removes the need for infrastructure management.
- **Serverless experience**: Developers can **quickly find and deploy** the right model for their use case.
- **Private customization**: Users can **fine-tune models with their own data** while maintaining privacy.

### Streaming Output Generation

Next, you explore how to use Amazon Bedrock's invoke_model_with_response_stream API to stream model outputs so users can consume outputs as they are generated. Rather than generating the full output at once, this API returns a ResponseStream that sends smaller output chunks from the model as they are produced. You can display these streaming outputs in a continuous, consumable view.

In [22]:
#invoke model with response stream
modelId = 'us.amazon.nova-2-lite-v1:0'
response = bedrock_client.invoke_model_with_response_stream(body=body, modelId=modelId, accept=accept, contentType=contentType)
stream = response.get('body')
output = list(stream)
output

[{'chunk': {'bytes': b'{"messageStart":{"role":"assistant"}}'}},
 {'chunk': {'bytes': b'{"contentBlockDelta":{"delta":{"text":"### Summary:\\n\\n"},"contentBlockIndex":0}}'}},
 {'chunk': {'bytes': b'{"contentBlockStop":{"contentBlockIndex":0}}'}},
 {'chunk': {'bytes': b'{"contentBlockDelta":{"delta":{"text":"AWS has launched"},"contentBlockIndex":1}}'}},
 {'chunk': {'bytes': b'{"contentBlockStop":{"contentBlockIndex":1}}'}},
 {'chunk': {'bytes': b'{"contentBlockDelta":{"delta":{"text":" **Amazon Bedrock**,"},"contentBlockIndex":2}}'}},
 {'chunk': {'bytes': b'{"contentBlockStop":{"contentBlockIndex":2}}'}},
 {'chunk': {'bytes': b'{"contentBlockDelta":{"delta":{"text":" a new service designed"},"contentBlockIndex":3}}'}},
 {'chunk': {'bytes': b'{"contentBlockStop":{"contentBlockIndex":3}}'}},
 {'chunk': {'bytes': b'{"contentBlockDelta":{"delta":{"text":" to make **"},"contentBlockIndex":4}}'}},
 {'chunk': {'bytes': b'{"contentBlockStop":{"contentBlockIndex":4}}'}},
 {'chunk': {'bytes': b

In [23]:
from IPython.display import display_markdown,Markdown,clear_output

In [28]:

modelId = 'us.amazon.nova-2-lite-v1:0'

response = bedrock_client.invoke_model_with_response_stream(
    body=body,
    modelId=modelId,
    accept=accept,
    contentType=contentType
)

stream = response.get('body')

output = []
i = 1

if stream:
    for event in stream:
        chunk = event.get('chunk')

        if chunk:
            chunk_obj = json.loads(chunk.get('bytes').decode())

            # Extract text from Nova streaming response
            content_block_delta = chunk_obj.get('contentBlockDelta')

            if content_block_delta:
                text = content_block_delta.get('delta', {}).get('text')

                if text:
                    clear_output(wait=True)
                    output.append(text)

                    display_markdown(
                        Markdown(''.join(output))
                    )

                    i += 1


### Summary:

AWS has launched **Amazon Bedrock**, a new service designed to make **foundation models (FMs)** from leading AI companies — including AI21 Labs, Anthropic, Stability AI, and Amazon — easily accessible through a simple API. Bedrock aims to **democratize access** to powerful generative AI models for all developers, enabling them to build and scale AI-driven applications efficiently.

Key features of Amazon Bedrock include:

- **Access to a variety of FMs** for both **text and image generation**, including **Amazon’s new Titan FMs** — two newly announced large language models (LLMs).
- **Serverless experience**: Users can quickly find and deploy the right model for their needs **without managing infrastructure**.
- **Private customization**: Developers can tailor models using their **own data**.
- **Seamless integration** with familiar **AWS tools** such as **Amazon SageMaker**, including features like **Experiments** for model testing and **Pipelines** for managing models at scale.

In essence, Bedrock simplifies the process of building generative AI applications by offering a **scalable, reliable, and secure** managed service that integrates smoothly with the AWS ecosystem.

You have now experimented with using the boto3 SDK to access the Amazon Bedrock API. This SDK provides basic programmatic access to Bedrock capabilities. By leveraging this API, you were able to implement two use cases: 1) Generating an entire text summary of AWS news content at once, and 2) Streaming the summary output in chunks for incremental processing.

### Try it yourself
- Change the prompts to your specific usecase and evaluate the output of different models.
- Play with the token length to understand the latency and responsiveness of the service.
- Apply different prompt engineering principles to get better outputs.

### Cleanup

You have completed this notebook. To move to the next part of the lab, do the following:

- Close this notebook file and continue with **Task2b.ipynb**.